In [1]:
#pip install selenium
#pip install webdriver_manager
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import unquote
from selenium.webdriver.support.ui import Select
import pandas as pd
import datetime 
from selenium.webdriver.common.keys import Keys
import warnings
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

In [2]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.get('https://parts.gmparts.com/')
wait=WebDriverWait(driver, 10)
### Please update capatche.

# MMY Extract

In [24]:
cols=['Sl.No.']
df = pd.DataFrame(columns=cols)
df
count=0

In [25]:
driver.find_element(By.ID,"view-my-garage-button").click()
try:
    driver.find_element(By.CSS_SELECTOR,'[data-dtm="recently searched"]').click()
except:
    pass

driver.find_element(By.ID,"YMM_YEAR").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{2001}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"YMM_MAKE").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{"GMC"}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"YMM_MODEL").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{"Savana 1500"}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"YMM_BODY").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{"3 Door - Standard Passenger Van"}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"prt-partial-fitment-search").click()
sleep(.5)
driver.find_element(By.ID,"YMM_WHEEL").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{"135.0"}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"YMM_TRIM").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{"SLT"}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"YMM_DRIVE").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{"RWD"}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"YMM_ENGINE").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[value="{"4.3L V6 GAS"}"]'))).click()
sleep(.5)
driver.find_element(By.ID,"mygarage-acYmmSearchSubmit").click()
sleep(4)
sb=driver.find_element(By.ID,"prt-search-by-input-field")
sb.click()
sb.send_keys("Spark Plug Wire Kit")
sb.send_keys(Keys.ENTER)

In [29]:
Names=driver.find_elements(By.ID,'prt-psr-card-title')
PartNumbers=driver.find_elements(By.ID,'prt-product-card-part-numbers')
Values=driver.find_elements(By.CLASS_NAME,'top-pricing-row')
types=driver.find_elements(By.ID,'parallelogram-widget')
FitmentNotes=driver.find_elements(By.CLASS_NAME,'card-body')
Links=driver.find_elements(By.ID,'card-link')

for Name, PN, Cost, type,  Link, fitment in zip(Names, PartNumbers, Values,types[3:], Links,FitmentNotes[4:]):
    if "Spark Plug Wire Kit" in Name.text or "Spark Plug Wire Set"in Name.text:
        df.loc[count,'Name']=Name.text
        df.loc[count,'GM Part Number']=PN.text.split("\n")[0].split("#")[1]
        try:
            df.loc[count,'ACDELCO Part Number']=PN.text.split("\n")[1].split("#")[1]
        except:
            df.loc[count,'ACDELCO Part Number']=""
        df.loc[count,'MSRP']=float(Cost.text.split("MSRP")[1].split("$")[1])
        df.loc[count,'Type']=(type.text)
        df.loc[count,'Links']=(Link.get_attribute("href").split("?")[0])
        if "Check Fitment Notes" in fitment.text:
            df.loc[count,'Fitment Notes']="Yes"
        else:
            df.loc[count,'Fitment Notes']="No"
        count=count+1
    else:
        pass

In [30]:
df

,Sl.No.,Name,GM Part Number,ACDELCO Part Number,MSRP,Type,Links,Fitment Notes
0,NaN,GM Genuine Parts Spark Plug Wire Kit,19433890,19433890,238.84,OE,https://parts.gmparts.com/product/gm-genuine-p...,No
1,NaN,ACDelco Gold Spark Plug Wire Set,88862384,9746KK,116.84,GOLD,https://parts.gmparts.com/product/acdelco-gold...,No


# Parts Extract

In [93]:
cols=['Sl.No.']
df_part = pd.DataFrame(columns=cols)
df_part
count1=0

In [95]:
ProductName="Coolant Hoses and Pipes"

In [4]:
search=driver.find_element(By.CSS_SELECTOR,'[id="prt-search-by-input-field"]')
search.send_keys(ProductName)
search.send_keys(Keys.ENTER)

In [89]:
for i in range(50):
    for i in range(3):
            driver.find_element(By.TAG_NAME,'body').send_keys(Keys.END) 
    driver.find_element(By.CSS_SELECTOR,'[data-dtm2="button:view more"]').click()
    sleep(5)

KeyboardInterrupt: 

In [92]:
print(len(driver.find_elements(By.CSS_SELECTOR,'[class="card-bottom-section card-body"]')))

2376


In [94]:
lists=driver.find_elements(By.CSS_SELECTOR,'[class="card-bottom-section card-body"]')
links=driver.find_elements(By.CSS_SELECTOR,'[class="stat-image-link"]')
for l,link in zip(lists,links[1:]):
    df_part.loc[count1,'Sl.No.']=count1+1
    df_part.loc[count1,'Name']=l.text.split("\n")[0]
    try:
        df_part.loc[count1,'GM Part Number']=l.text.split("GM Part # ")[1].split("\n")[0]
    except:
        df_part.loc[count1,'GM Part Number']=""
    try:
        df_part.loc[count1,'AC Delco Part Number']=l.text.split("ACDelco Part # ")[1].split("\n")[0]
    except:
        df_part.loc[count1,'AC Delco Part Number']=""
    try:
        df_part.loc[count1,'Starting Price [$]']=float(l.text.split("Starting at \n$")[1].split("\n")[0])
    except:
        df_part.loc[count1,'Starting Price [$]']=0
    try:
        df_part.loc[count1,'MSRP Price [$]']=float(l.text.split("MSRP \n$")[1].split("\n")[0])
    except:
        df_part.loc[count1,'MSRP Price [$]']=0
    df_part.loc[count1,'Link']=link.get_attribute('href').split("?")[0]
    count1=count1+1


In [96]:
df_part

,Sl.No.,Name,GM Part Number,AC Delco Part Number,Starting Price [$],MSRP Price [$],Link
0,1,GM Genuine Parts Engine Coolant Water Outlet,25193922,15-11105,89.38,89.38,https://parts.gmparts.com/product/gm-genuine-p...
1,2,GM Genuine Parts Multi-Purpose Wiring Connector,55354565,55354565,8.20,8.20,https://parts.gmparts.com/product/gm-genuine-p...
2,3,GM Genuine Parts Engine Coolant Hose,13251447,13251447,45.72,45.72,https://parts.gmparts.com/product/gm-genuine-p...
3,4,GM Genuine Parts HVAC Heater Hose,84097808,84097808,25.63,25.63,https://parts.gmparts.com/product/gm-genuine-p...
4,5,GM Genuine Parts Compressed Natural Gas (CNG) ...,22990993,22990993,8.43,8.43,https://parts.gmparts.com/product/gm-genuine-p...
...,...,...,...,...,...,...,...
2371,2372,ACDelco GM Original Equipment Radiator Inlet Hose,85643556,,52.77,52.77,https://parts.gmparts.com/product/acdelco-gm-o...
2372,2373,GM Genuine Parts Radiator Surge Tank Outlet Hose,85643558,,110.25,110.25,https://parts.gmparts.com/product/gm-genuine-p...
2373,2374,GM Genuine Parts Radiator Surge Tank Inlet Hose,85643575,,100.77,100.77,https://parts.gmparts.com/product/gm-genuine-p...
2374,2375,GM Genuine Parts Radiator Outlet Hose,94012601,,84.82,84.82,https://parts.gmparts.com/product/gm-genuine-p...


In [97]:
with pd.ExcelWriter(fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\OE-Parts\Output\OEM_Parts_{ProductName}.xlsx') as writer:  # doctest: +SKIP
    df_part.to_excel(writer,index=False, sheet_name='Raw')